In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
# Transfomer Decoder:
# 1. token embedding
# 2. position embedding
# 3. Multihead attention
# 4. Feed forward Nural network
# 5. attention +FFN ：decoder block
# 6. 多个decoder堆叠起来
# 7.laynorm、activation、残差连接

In [3]:
class FeedForwardNeuralNetwork(nn.Module):
    def __init__(self, d_model, d_ff):
        super(FeedForwardNeuralNetwork, self).__init__()
        # Layer normalization
        self.layer_norm = nn.LayerNorm(d_model)
        # Linear projection layers
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        # Activation function
        self.activation = nn.GELU()
        
    def forward(self, x):
        """
        Forward pass of the feed-forward block.

        Args:
            x (Tensor): Input tensor of shape [batch_size, seq_len, hidden_size].

        Returns:
            Tensor: Output tensor of the same shape as input.
        """
        residual = x
        output = self.layer_norm(x)                 # [batch_size, seq_len, hidden_size]
        output = self.linear1(output)               # [batch_size, seq_len, hidden_size * 4]
        output = self.activation(output)            # [batch_size, seq_len, hidden_size * 4]
        output = self.linear2(output)               # [batch_size, seq_len, hidden_size]
        
        return residual + output


In [4]:
batch_size, seq_len, hidden_size = 16, 10, 768

x = torch.randn(batch_size, seq_len, hidden_size)

ffn = FeedForwardNeuralNetwork(768, 768*4)

output = ffn(x)

print(f"x size is {x.size()}")
print(f"output size is {output.size()}")
print(f"output is {output[0]}")

x size is torch.Size([16, 10, 768])
output size is torch.Size([16, 10, 768])
output is tensor([[ 0.6165, -0.7426, -0.1334,  ..., -0.1296,  0.9605,  0.4799],
        [-1.3581, -0.0603,  0.5108,  ...,  0.5609, -0.1649, -0.5700],
        [-1.5393,  0.9488,  1.2312,  ..., -0.8807,  1.7546, -0.1277],
        ...,
        [-0.9493, -0.0292, -0.2938,  ..., -1.3225, -0.8133,  0.2452],
        [ 1.4672,  1.1025, -0.1753,  ..., -1.1861, -1.1136,  0.2404],
        [-0.5821,  1.3923, -2.2210,  ..., -2.0225, -0.7759,  0.8451]],
       grad_fn=<SelectBackward0>)


In [5]:
class ScaledDotProductAttention(nn.Module):
    """Scaled Dot-Product Attention
    
    Computes the attention weights using the formula:
        Attention(Q, K, V) = softmax((Q * K^T) / sqrt(d_model))
    """
    def __init__(self):
        super(ScaledDotProductAttention, self).__init__()

    def forward(self, query, key, value, mask=None):
        """
        Compute attention weights and output.

        Args:
            query: Query tensor of shape [batch_size, seq_len, d_model]
            key: Key tensor of shape [batch_size, seq_len, d_model]
            value: Value tensor of shape [batch_size, seq_len, d_model]
            mask: Optional mask tensor (same shape as attention scores)

        Returns:
            output: Attention output tensor [batch_size, seq_len, d_model]
            attention_weights: Attention weights [batch_size, seq_len, seq_len]
        """
        d_q = query.size()[-1]

        # Compute scaled dot-product attention scores - [batch_size, seq_len, seq_len]
        scores = torch.matmul(query, key.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_q, dtype=torch.float32))

        # Apply mask (if provided)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        # Apply softmax to get attention weights
        attention_weights = torch.softmax(scores, dim=-1)

        # Compute the final output  - - [batch_size, seq_len, d_model]
        output = torch.matmul(attention_weights, value)

        return output, attention_weights

In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        self.d_model = d_model
        self.num_heads = num_heads
        
        super(MultiHeadAttention, self).__init__()
        assert self.d_model%self.num_heads==0, "d_model must be divisible by num_heads"
        
        self.head_dim = self.d_model//self.num_heads
        
        self.query_proj = nn.Linear(d_model, d_model)
        self.key_proj = nn.Linear(d_model, d_model)
        self.value_proj = nn.Linear(d_model, d_model)
        
        self.out_proj = nn.Linear(d_model, d_model)
        self.attention = ScaledDotProductAttention()
    
    def split_heads(self, x):
        '''
        
        :param x: [batch_size, seq_len, d_model]
        :return: 
            [batch_size, num_heads, seq_len, head_dim]
        '''
        batch_size, seq_len, d_model = x.size()
        assert  d_model==self.head_dim*self.num_heads, f"input must in dim {self.num_heads*self.head_dim} but input dim is {d_model}"
        
        return x.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1,2)
    def combine_heads(self,x):
        '''
        
        :param x:  [batch_size, num_heads, seq_len, head_dim]
        :return:  [batch_size, seq_len, num_heads*head_dim]
        '''
        batch_size, num_heads, seq_len, head_dim = x.size()
        
        return x.transpose(1,2).contiguous().view(batch_size, seq_len,num_heads*head_dim)
        
    def forward(self, x, mask=None):
        
        query = self.query_proj(x)
        key = self.key_proj(x)
        value = self.value_proj(x)
        
        splited_query = self.split_heads(query)
        splited_key = self.split_heads(key)
        splited_value = self.split_heads(value)
        
        output, scores =self.attention(splited_query, splited_key, splited_value, mask) 
        output = self.combine_heads(output)
        
        return self.out_proj(output), scores

In [7]:
class TransformerDecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super(TransformerDecoderBlock, self).__init__()
        # Core modules
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.feedForward = FeedForwardNeuralNetwork(d_model, d_ff)
        # Final layer normalization
        self.layer_norm = nn.LayerNorm(d_model)
        
    def forward(self, x, attn_mask=None):
        """
        Forward pass for a single Transformer decoder block.

        Args:
            x (Tensor): Input tensor of shape [batch_size, seq_len, d_model].
            attn_mask (Tensor, optional): Attention mask to prevent attending to certain positions 
                                          (e.g., future tokens during autoregressive decoding).

        Returns:
            Tuple[Tensor, Tensor]:
                - output: The processed tensor after attention, feed-forward network, and normalization.
                - attn_weights: Attention weights from the multi-head attention layer.
        """
        # Multi-head self-attention
        attn_output, attn_weights = self.attention(x, attn_mask)

        # Feed-forward network with residual connection
        ff_output = self.feedForward(x + attn_output)

        # Apply final layer normalization
        output = self.layer_norm(ff_output)
        
        return output, attn_weights


In [8]:
batch_size, seq_len, hidden_size = 16, 10, 768

x = torch.randn(batch_size, seq_len, hidden_size)

tdb = TransformerDecoderBlock(hidden_size, 12, hidden_size*4)

output, attn_weights = tdb(x)
print(f"x size is {x.size()}")
print(f"output size is {output.size()}")
print(f"output is {output[0]}")

x size is torch.Size([16, 10, 768])
output size is torch.Size([16, 10, 768])
output is tensor([[-0.4039, -0.7145, -0.0245,  ...,  1.1260,  1.7852, -1.5609],
        [-0.1983,  0.0236, -0.9027,  ..., -1.1018,  0.4945, -1.4249],
        [ 0.7095,  1.1802,  1.1961,  ..., -0.3822,  0.2406,  0.4434],
        ...,
        [-1.0423,  2.4532, -0.3434,  ..., -0.4855,  0.7532,  1.8846],
        [ 2.5117, -0.1292, -0.5675,  ...,  1.6438,  0.4477,  0.2887],
        [-0.9010, -1.8039, -0.4816,  ..., -1.0859, -0.4233, -0.3127]],
       grad_fn=<SelectBackward0>)


In [9]:
import math

In [10]:
class PositionalEncoding(nn.Module):
    """Positional Encoding Module (supports dynamic sequence lengths)"""
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # [1, max_len, d_model]

    def forward(self, x):
        # Dynamically obtain positional encoding
        position_emb = self.pe[:, :x.size(1)]
        return x + position_emb  # [batch, seq_len, d_model]


## Transformer Implementation

In [11]:
class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model, max_len, num_layers, num_heads, d_ff):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len)
        
        # Stack decoder blocks
        self.layers = nn.ModuleList([
            TransformerDecoderBlock(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.output_layer = nn.Linear(d_model, vocab_size, bias=False)
        
        # Tied embeddings
        self.output_layer.weight = self.token_embedding.weight
        
        self.init_weights()

    def init_weights(self):
        nn.init.normal_(self.token_embedding.weight, std=0.02)
        
        # Initialize each layer
        for layer in self.layers:
            nn.init.xavier_normal(layer.attention.query_proj.weight)
            nn.init.xavier_normal(layer.attention.key_proj.weight)
            nn.init.xavier_normal(layer.attention.value_proj.weight)
            
            nn.init.kaiming_normal(layer.feedForward.linear1.weight)
            nn.init.kaiming_uniform(layer.feedForward.linear2.weight)
    
    def create_causal_mask(self, seq_len):
        mask = torch.tril(torch.ones(seq_len, seq_len))
        return mask
    
    def forward(self, input_ids):
        '''
        :param x: [batch_size, seq_len]  
        :return: 
        '''
        _, seq_len = input_ids.size()
        
        # Embedding
        embeddings = self.token_embedding(input_ids)  # [batch_size, seq_len, d_model]
        pos_embedding = self.pos_encoder(embeddings)
        
        hidden_states = embeddings + pos_embedding
        mask = self.create_causal_mask(seq_len)

        # Pass through all Transformer decoder blocks
        all_attn_weights = []
        for layer in self.layers:
            hidden_states, attn_weights = layer(hidden_states, mask)
            all_attn_weights.append(attn_weights)
        
        hidden_states = self.final_norm(hidden_states)
        
        logits = self.output_layer(hidden_states)
        
        return logits, all_attn_weights


In [12]:
model = TransformerDecoder(
    vocab_size=500, d_model=256, max_len=128, num_layers=12, num_heads=8, d_ff=256*4)

/var/folders/7x/tfwsytqd3yjccjl53cm75j700000gn/T/ipykernel_14729/1372856348.py:25: FutureWarning: `nn.init.xavier_normal` is now deprecated in favor of `nn.init.xavier_normal_`.
  nn.init.xavier_normal(layer.attention.query_proj.weight)
/var/folders/7x/tfwsytqd3yjccjl53cm75j700000gn/T/ipykernel_14729/1372856348.py:26: FutureWarning: `nn.init.xavier_normal` is now deprecated in favor of `nn.init.xavier_normal_`.
  nn.init.xavier_normal(layer.attention.key_proj.weight)
/var/folders/7x/tfwsytqd3yjccjl53cm75j700000gn/T/ipykernel_14729/1372856348.py:27: FutureWarning: `nn.init.xavier_normal` is now deprecated in favor of `nn.init.xavier_normal_`.
  nn.init.xavier_normal(layer.attention.value_proj.weight)
/var/folders/7x/tfwsytqd3yjccjl53cm75j700000gn/T/ipykernel_14729/1372856348.py:29: FutureWarning: `nn.init.kaiming_normal` is now deprecated in favor of `nn.init.kaiming_normal_`.
  nn.init.kaiming_normal(layer.feedForward.linear1.weight)
/var/folders/7x/tfwsytqd3yjccjl53cm75j700000gn/T/ipy

In [13]:
batch_size, seq_len = 16,10

input_ids = torch.randint(0,500,(batch_size, seq_len))

print(f"input size is {input_ids.size()}")
print(f"input is {input_ids}")

output, _ = model(input_ids)

print(f"output size is {output.size()}")
print(f"output is {output[-1]}")

input size is torch.Size([16, 10])
input is tensor([[217, 175, 167, 292, 403, 206, 428, 137, 212, 367],
        [406, 466, 152, 189, 105, 164, 295, 417, 163, 262],
        [ 82, 243, 122, 469, 397, 363, 169, 423,  26,   0],
        [276, 475, 263, 486, 299, 239, 208, 399,  48, 331],
        [103, 218, 240,  50, 490, 189, 237,  75, 233, 188],
        [ 10, 383, 274, 292, 196, 374,  38, 123, 387,  11],
        [436, 164, 174, 217,  93, 317, 339, 462,  68, 268],
        [ 10, 430,  50, 489, 286, 147, 215, 250, 140,  86],
        [ 69, 418, 463, 341, 164, 310, 117,  35, 469,   2],
        [154, 409, 289, 290, 374, 398, 348, 431, 224, 137],
        [176,  57, 245, 407, 350, 407,  28, 405, 396, 406],
        [407, 391, 128, 456, 379, 400, 312, 100, 302,  59],
        [199,  64, 420, 343, 230, 100, 123,  49, 319, 318],
        [225, 203, 348, 493, 335, 160, 197, 353, 161, 379],
        [352, 270, 283, 489, 455,  58, 158, 221, 486, 357],
        [476, 333, 472,  10, 270, 246, 457, 106,  54, 16